# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

# Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")

# Silver Transformations

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Customer ID Cleanup

In [0]:
df = df.withColumn(
    "CID",
    F.when(col("CID").startswith("NAS"),
           F.substring(col("CID"), 4, F.length(col("CID"))))
        .otherwise(col("CID"))
)

## Birthdate Validation

In [0]:
df = df.withColumn(
    "BDATE",
    F.when(col("BDATE") > F.current_date(), None)
    .otherwise(col("BDATE"))
)

## Gender Normalization

In [0]:
df = df.withColumn(
    "GEN",
    F.when(F.upper(col("GEN")).isin("F", "FEMALE"), "Female")
    .when(F.upper(col("GEN")).isin("M", "MALE"), "Male")
    .otherwise("n/a")
)

## Renaming Columns

In [0]:
RENAME_MAP = {
    "CID": "customer_number",
    "BDATE": "birth_date",
    "GEN": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.display()

# Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")

In [0]:
%sql
select * from workspace.silver.erp_customers LIMIT 10